**⚠️ Technical Note:** This notebook was developed in a Google Colab environment. The raw bioclimatic and GEDI/VHM datasets are hosted in a private Google Drive directory. To replicate this study, users must provide their own raster assets or contact the author for access to the standardized 1km² Parquet files.

# 04: IPCC CMIP6 Data Acquisition & Aggregation
**Project:** A Validated Predictive Framework for Climate-Smart Reforestation in Armenia

**Author:** Narek Ohanyan

In [1]:
!pip install intake-esm gcsfs xarray dask netCDF4 zarr cftime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.4/115.4 kB 9.

In [2]:
import intake
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Open the Pangeo CMIP6 catalog
url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
col = intake.open_esm_datastore(url)

# Define our query parameters
query = dict(
    experiment_id=['historical', 'ssp126', 'ssp245', 'ssp370', 'ssp585'],
    table_id='day',
    variable_id=['tasmax', 'pr', 'hurs'],
    member_id='r1i1p1f1',
    source_id='MPI-ESM1-2-LR'
)

# Search the catalog
cat_subset = col.search(**query)
print(f"Datasets found: {len(cat_subset.df)}")

# Load data into an xarray dataset dictionary (lazy loading via dask)
# ADDED: storage_options={'token': 'anon'} to bypass Google credential checks
dset_dict = cat_subset.to_dataset_dict(
    zarr_kwargs={'consolidated': True},
    storage_options={'token': 'anon'}
)

Datasets found: 15

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


<div><progress max="5" value="5"></progress> 100.00% [5/5 00:08&lt;00:00]</div>

In [5]:
def subset_armenia(ds):
    """Subsets the dataset to the approximate bounding box of Armenia."""
    lat_bnds, lon_bnds = [38.8, 41.3], [43.4, 46.6]

    # Handle longitude conventions (0-360 vs -180 to 180)
    if ds.lon.max() > 180:
        ds = ds.assign_coords(lon=(ds.lon + 180) % 360 - 180)
        ds = ds.sortby(ds.lon)

    ds_sub = ds.sel(lat=slice(lat_bnds[0], lat_bnds[1]), lon=slice(lon_bnds[0], lon_bnds[1]))
    return ds_sub.mean(dim=['lat', 'lon']) # Return spatial average

def calculate_vpd(tasmax_k, hurs):
    """
    Calculates Vapor Pressure Deficit (VPD) in kPa.
    tasmax_k: Max temp in Kelvin
    hurs: Relative humidity in %
    """
    # Convert Temp to Celsius
    t_c = tasmax_k - 273.15

    # Saturation Vapor Pressure (es)
    es = 0.6108 * np.exp((17.27 * t_c) / (t_c + 237.3))

    # VPD calculation
    vpd = es * (1 - (hurs / 100))
    return vpd

In [11]:
import pandas as pd
from IPython.display import display

# Define time periods
periods = {
    'baseline': slice('1995', '2014'),
    'mid_term': slice('2041', '2060'),
    'long_term': slice('2081', '2100')
}

results = {}

print("Subsetting spatial domain and calculating variable conversions...")
for key, ds in dset_dict.items():
    exp = ds.attrs['experiment_id']

    # Subset to Armenia
    ds_armenia = subset_armenia(ds)

    # CRITICAL FIX 1: Isolate the Vegetation Period (May - September)
    # This guarantees the mean() functions later only average summer climate.
    ds_armenia_gs = ds_armenia.where(ds_armenia['time'].dt.month.isin([5, 6, 7, 8, 9]), drop=True)

    # Calculate VPD daily (Formula natively outputs in kPa)
    if 'tasmax' in ds_armenia_gs and 'hurs' in ds_armenia_gs:
        ds_armenia_gs['vpd'] = calculate_vpd(ds_armenia_gs['tasmax'], ds_armenia_gs['hurs'])

    # Convert PR from flux to 5-month cumulative mm
    # 1 average month = 30.4375 days = 2,629,800 seconds
    # 5 months = 13,149,000 seconds
    if 'pr' in ds_armenia_gs:
        ds_armenia_gs['pr_mm_season'] = ds_armenia_gs['pr'] * (2629800 * 5)

    results[exp] = ds_armenia_gs

print("Computing historical baseline mean...")
hist_ds = results['historical'].sel(time=periods['baseline']).mean(dim='time').compute()

# Scenarios to analyze
scenarios = ['ssp126', 'ssp245', 'ssp370', 'ssp585']
future_periods = ['mid_term', 'long_term']
period_labels = {'mid_term': 'Medium-Term (2041–2060)', 'long_term': 'Long-Term (2081–2100)'}

table_data = []

print("Computing future scenarios and calculating deltas...")
for ssp in scenarios:
    if ssp not in results:
        continue

    for period_key in future_periods:
        future_ds = results[ssp].sel(time=periods[period_key]).mean(dim='time').compute()

        # 1. Temperature delta (°C) -> mathematically identical to Kelvin delta
        delta_tmax_c = float(future_ds['tasmax'].values - hist_ds['tasmax'].values)

        # 2. Precipitation absolute delta in cumulative mm/season
        delta_pr_mm = float(future_ds['pr_mm_season'].values - hist_ds['pr_mm_season'].values)

        # 3. VPD absolute delta natively in kPa
        delta_vpd_kpa = float(future_ds['vpd'].values - hist_ds['vpd'].values)

        table_data.append({
            'Time Horizon': period_labels[period_key],
            'Pathway': ssp.upper(),
            'ΔTmax [°C]': round(delta_tmax_c, 2),
            'ΔP [mm]': round(delta_pr_mm, 2),
            'ΔVPD [kPa]': round(delta_vpd_kpa, 3)
        })

# Generate and display the table
print("\n--- Streamlit Projection Matrix Data ---")
df_results = pd.DataFrame(table_data)

# Sort logically to match the nested dictionary structure in the app
df_results = df_results.sort_values(by=['Time Horizon', 'Pathway'])
display(df_results)

Subsetting spatial domain and calculating variable conversions...
Computing historical baseline mean...
Computing future scenarios and calculating deltas...

--- Streamlit Projection Matrix Data ---


,Time Horizon,Pathway,ΔTmax [°C],ΔP [mm],ΔVPD [kPa]
1,Long-Term (2081–2100),SSP126,1.08,-6.39,0.194
3,Long-Term (2081–2100),SSP245,2.41,-22.21,0.417
5,Long-Term (2081–2100),SSP370,5.44,-67.50,1.073
7,Long-Term (2081–2100),SSP585,5.72,-49.17,1.045
0,Medium-Term (2041–2060),SSP126,1.02,-9.70,0.168
2,Medium-Term (2041–2060),SSP245,1.79,-20.47,0.271
4,Medium-Term (2041–2060),SSP370,2.71,-45.21,0.511
6,Medium-Term (2041–2060),SSP585,1.97,-30.99,0.277
